# Daudz aģentu sistēma klientu atbalsta automatizācijai

In [ ]:
!pip install crewai crewai_tools

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import userdata
import os

os.environ['GEMINI_API_KEY']=userdata.get('GOOGLE_API_KEY')

In [ ]:
from crewai import Agent, Task, Crew

## Lomu spēlēšana un sadarbība

In [ ]:
support_agent = Agent(
    role="Senior Support Representative",
	goal="Be the most friendly and helpful "
        "support representative in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        " are now working on providing "
		"support to {customer}, a super important customer "
        " for your company."
		"You need to make sure that you provide the best support!"
		"Make sure to provide full complete answers, "
        " and make no assumptions."
	),
	allow_delegation=False,
	llm = "gemini/gemini-1.5-flash",
	verbose=True
)

* Neiestatot allow_delegation=False, allow_delegation
iegūst tā noklusēto vērtību, kas ir True.
* Tas nozīmē, ka aģents var deleģēt savu darbu citam aģentam, kuram ir labākas prasmes veikt konkrētu uzdevumu.

In [ ]:
support_quality_assurance_agent = Agent(
	role="Support Quality Assurance Specialist",
	goal="Get recognition for providing the "
    "best support quality assurance in your team",
	backstory=(
		"You work at crewAI (https://crewai.com) and "
        "are now working with your team "
		"on a request from {customer} ensuring that "
        "the support representative is "
		"providing the best support possible.\n"
		"You need to make sure that the support representative "
        "is providing full"
		"complete answers, and make no assumptions."
	),
	llm = "gemini/gemini-1.5-flash",
	verbose=True
)

* **Lomu spēlēšana**: Abiem aģentiem ir piešķirta loma, mērķis un pamata stāsts.
* **Uzmanība**: Abiem aģentiem ir iedota norāde iedzīvoties viņu spēlētajās lomās.
* **Sadarbība**: Klientu atbalsta kvalitātes nodrošināšanas aģents var deleģēt darbu atpakaļ atbalsta aģentam, ļaujot šiem aģentiem strādāt kopā.

## Rīki

In [ ]:
from crewai_tools import SerperDevTool, \
                         ScrapeWebsiteTool, \
                         WebsiteSearchTool

* Izveidojiet dokumentu skrāpēšanas rīku.
* Rīks skrāpēs vienu lapu (tikai vienu URL) no CrewAI dokumentācijas.

In [ ]:
docs_scrape_tool = ScrapeWebsiteTool(
    website_url="https://docs.crewai.com/how-to/Creating-a-Crew-and-kick-it-off/"
)

Dažādi veidi, kā piešķirt rīkus aģentiem

* Aģenta līmenis: Aģents var izmantot rīku(-us) jebkurā uzdevumā, ko tas veic.
* Uzdevuma līmenis: Aģents izmantos rīku(-us) tikai tad, kad veiks konkrēto uzdevumu.
**Piezīme**: Uzdevuma rīki pārspēj aģenta rīkus.

In [ ]:
inquiry_resolution = Task(
    description=(
        "{customer} just reached out with a super important ask:\n"
	    "{inquiry}\n\n"
        "{person} from {customer} is the one that reached out. "
		"Make sure to use everything you know "
        "to provide the best support possible."
		"You must strive to provide a complete "
        "and accurate response to the customer's inquiry."
    ),
    expected_output=(
	    "A detailed, informative response to the "
        "customer's inquiry that addresses "
        "all aspects of their question.\n"
        "The response should include references "
        "to everything you used to find the answer, "
        "including external data or solutions. "
        "Ensure the answer is complete, "
		"leaving no questions unanswered, and maintain a helpful and friendly "
		"tone throughout."
    ),
	tools=[docs_scrape_tool],
    agent=support_agent,
)

In [ ]:
quality_assurance_review = Task(
    description=(
        "Review the response drafted by the Senior Support Representative for {customer}'s inquiry. "
        "Ensure that the answer is comprehensive, accurate, and adheres to the "
		"high-quality standards expected for customer support.\n"
        "Verify that all parts of the customer's inquiry "
        "have been addressed "
		"thoroughly, with a helpful and friendly tone.\n"
        "Check for references and sources used to "
        " find the information, "
		"ensuring the response is well-supported and "
        "leaves no questions unanswered."
    ),
    expected_output=(
        "A final, detailed, and informative response "
        "ready to be sent to the customer.\n"
        "This response should fully address the "
        "customer's inquiry, incorporating all "
		"relevant feedback and improvements.\n"
		"Don't be too formal, we are a chill and cool company "
	    "but maintain a professional and friendly tone throughout."
    ),
    agent=support_quality_assurance_agent,
)


### Komandas izveide

In [ ]:
crew = Crew(
  agents=[support_agent, support_quality_assurance_agent],
  tasks=[inquiry_resolution, quality_assurance_review],
  verbose=True
)

### Komandas palaišana

In [ ]:
inputs = {
    "customer": "DeepLearningAI",
    "person": "Andrew Ng",
    "inquiry": "I need help with setting up a Crew "
               "and kicking it off, specifically "
               "how can I add memory to my crew? "
               "Can you provide guidance?"
}
result = crew.kickoff(inputs=inputs)

- Atspoguļot galīgo rezultātu kā Markdown.

In [ ]:
from IPython.display import Markdown
Markdown(result.raw)